# Figure 2 — Distinct perturbation programs converge on failed endothelial differentiation

**Implemented here:** panels a–b. Panels c–f remain in their dedicated source/asset workflows.

Canonical source inspected: `programDE/notebooks/program_level_temporal_map_notebooks/02_gene_program_effect_temporal_heatmap.ipynb` on `sherlock-sync-20261013-070509`. The port preserves the source convention of clustering target effect vectors with correlation distance and average linkage.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from src.cc_perturbseq.figure_notebook import (
    get_path, initialize_figure, load_figure_manifest, panel_directory, write_receipt,
)
from src.cc_perturbseq.program_panels import *

def export_figure(fig, stem, panel):
    out = panel_directory(ctx, panel)
    paths = {}
    for suffix, kwargs in [("pdf", {}), ("svg", {}), ("png", {"dpi": 300})]:
        path = out / f"{stem}.{suffix}"
        fig.savefig(path, **kwargs)
        paths[suffix] = str(path)
    return paths

FIGURE = load_figure_manifest("Figure_02")
ctx = initialize_figure(FIGURE)
ctx["input_audit"]


## Load programDE effects

The loader accepts either the consolidated long programDE export or a directory of per-day result tables. It does not recalculate CRT effects.


In [ ]:
effect_path = get_path(ctx["paths"], "programde.effect_matrix_all_days")
q_path = get_path(ctx["paths"], "programde.qvalue_matrix_all_days", required=False)
filename = config_value(ctx["paths"], "programde.effects_filename", required=False)
effects = load_program_effects(effect_path, q_path, filename=filename)
effects.groupby(["day", "program"]).size().unstack(fill_value=0)


## Panel a — Pairwise target correlation across programs and time points


In [ ]:
TARGET_GROUPS = {
    "BMP/TGFβ": ["BMPR1A", "SMAD1", "SMAD5", "SMAD4"],
    "Mesoderm": ["MIXL1", "EOMES"],
    "LM + EC": ["KDR", "KCTD10", "PRKCD", "ETV2", "PLCG1"],
    "Hippo": ["TAOK1", "AMOTL2", "LATS2", "NF2"],
}
fig, correlation, target_order = plot_target_effect_correlation(
    effects, target_groups=TARGET_GROUPS, min_features=10
)
outputs = export_figure(fig, "Figure_02a_perturbation_effect_correlation_heatmap", "a")
source_tables = save_source_tables(
    {"target_correlation": correlation, "target_order": pd.DataFrame({"target": target_order})},
    panel_directory(ctx, "a"), "Figure_02a"
)
write_receipt(ctx, "a", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"effects": str(effect_path)}, outputs={**outputs, **source_tables}, parameters={"distance": "correlation", "linkage": "average", "min_features": 10})
plt.show()


## Panel b — Selected regulator-by-program effects and Day 3 EC output

The program set follows the current curated developmental axes. Exact columns can be overridden in this cell after the final program-annotation table is frozen.


In [ ]:
EC_path = get_path(ctx["paths"], "ccperturbseq.ec_fraction_table")
ec_output = load_ec_output(EC_path)
PROGRAMS_BY_DAY = {
    "D1": [28, 13, 30],
    "D2": [15, 23, 14, 17, 19],
    "D3": [2, 3, 8, 11, 34, 27, 35],
}
fig, values, qvalues = plot_selected_program_effects(
    effects, ec_output,
    target_groups=TARGET_GROUPS,
    programs_by_day=PROGRAMS_BY_DAY,
)
outputs = export_figure(fig, "Figure_02b_gene_program_effects_heatmap", "b")
source_tables = save_source_tables({"effects": values, "qvalues": qvalues}, panel_directory(ctx, "b"), "Figure_02b")
write_receipt(ctx, "b", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"effects": str(effect_path), "ec_output": str(EC_path)}, outputs={**outputs, **source_tables}, parameters={"programs_by_day": PROGRAMS_BY_DAY, "target_groups": TARGET_GROUPS})
plt.show()


## Remaining panels

Panels c–f are intentionally not reconstructed here because they use a manual schematic, guide-level EC-fraction plotting, UMAP expression/program layers, and carrier-seeded neighborhood outputs rather than the programDE temporal-map scripts.
